# RAGxiv — Controlled RAG Experiments (LlamaIndex + QASPER)

**Jupyter is the source of truth:** notebook cells generate `ragxiv_lab.py` and `app.py` and can launch Streamlit directly. The experiment grid varies one family of techniques at a time, with **two variants per tunable method** and **no execution time cutoff**.

Track: Precision@K, Recall@K, Hit@K, MRR, MAP, nDCG, query latency, plus optional groundedness, context relevance, answer relevance and RAGAS evaluation. The baseline uses QASPER's original paragraph evidence labels.


## 0. Setup

In [1]:
import importlib.metadata as md
import instructor
print("RAGAS:", md.version("ragas"))
print("Instructor:", md.version("instructor"))
print("OpenAI-compatible client:", md.version("openai"))
print("No Gemini API key required for local RAGAS judging.")


RAGAS: 0.4.4
Instructor: 1.17.0
OpenAI-compatible client: 3.3.0
No Gemini API key required for local RAGAS judging.


In [2]:
from pathlib import Path
import sys, json, pandas as pd
print('Python:',sys.version)
print('Working directory:',Path.cwd())
Path('artifacts/runs').mkdir(parents=True,exist_ok=True)

Python: 3.14.8 (main, Sep 30 2026, 17:55:09) [Clang 21.0.0 (clang-2100.3.34.2)]
Working directory: /Users/aryan.agarwal/RAGxiv


## 1. Write the LlamaIndex RAG experiment engine

In [3]:
%%writefile ragxiv_lab.py
"""RAGxiv: QASPER research-paper RAG experiments (paper-scoped evaluation)."""
from __future__ import annotations
import json, math, re, tarfile, time, urllib.request, os, hashlib, itertools
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional

ROOT = Path.cwd()
ART = ROOT / 'artifacts'
URL = 'https://qasper-dataset.s3.us-west-2.amazonaws.com/qasper-train-dev-v0.3.tgz'
EMBED_NAME = 'BAAI/bge-small-en-v1.5'


def norm(s):
    return re.sub(r'\s+', ' ', str(s or '')).strip()


def load_papers(split='dev', limit=12):
    """Official JSON, not obsolete HuggingFace dataset-script loader."""
    assert split in ('train','dev'), 'Do not tune on QASPER test.'
    ART.mkdir(exist_ok=True, parents=True)
    target = ART / f'qasper-{split}-v0.3.json'
    if not target.exists():
        archive = ART / 'qasper-train-dev-v0.3.tgz'
        if not archive.exists():
            urllib.request.urlretrieve(URL, archive)
        with tarfile.open(archive, 'r:gz') as tf:
            member = next((m for m in tf.getmembers() if m.isfile() and Path(m.name).name == target.name), None)
            if member is None: raise FileNotFoundError(f'{target.name} not found in official archive')
            with tf.extractfile(member) as inp, target.open('wb') as out:
                import shutil
                shutil.copyfileobj(inp,out)
    data = json.loads(target.read_text(encoding='utf-8'))
    return dict(list(data.items())[:limit]) if limit else data


def paragraphs(paper):
    """Returns stable IDs tied to original evidence units."""
    result=[]
    abstract=paper.get('abstract') or ''
    if isinstance(abstract,list): abstract=' '.join(map(str,abstract))
    if norm(abstract):result.append(('abstract', 'Abstract', norm(abstract)))
    ft=paper.get('full_text') or []
    # QASPER's official JSON uses a list of section dictionaries. Some converted
    # variants use a column-oriented dict; accept either representation.
    if isinstance(ft, list):
        sections=ft
    elif isinstance(ft, dict):
        names=ft.get('section_name',[])
        contents=ft.get('paragraphs',[])
        sections=[{'section_name':names[i] if i<len(names) else f'Section {i+1}',
                   'paragraphs':ps} for i,ps in enumerate(contents)]
    else:
        raise TypeError(f'Unexpected full_text format: {type(ft)}')
    for si, section in enumerate(sections):
        if not isinstance(section,dict): continue
        name=section.get('section_name') or f'Section {si+1}'
        parts=section.get('paragraphs') or []
        if isinstance(parts,str): parts=[parts]
        for pi,part in enumerate(parts):
            if isinstance(part,str) and norm(part):
                result.append((f's{si}_p{pi}',str(name),norm(part)))
    return result


def annotations(paper_id,paper):
    qas=paper.get('qas') or {}
    if isinstance(qas,list): rows=qas
    else:
        qs=qas.get('question',[]); ids=qas.get('question_id',[]); ans=qas.get('answers',[])
        rows=[{'question':q, 'question_id':ids[i] if i<len(ids) else f'{paper_id}_{i}',
               'answers':ans[i] if i<len(ans) else []} for i,q in enumerate(qs)]
    out=[]
    for qi,qa in enumerate(rows):
        aa=qa.get('answers') or []
        if isinstance(aa,dict):
            aa=[{'answer': item} for item in aa.get('answer',[])]
        ev=[]; refs=[]; unanswerable=False
        for a in aa:
            d=a.get('answer',a) if isinstance(a,dict) else {}
            if not isinstance(d,dict):continue
            ev.extend(d.get('evidence') or [])
            if d.get('unanswerable'): unanswerable=True
            if d.get('free_form_answer'):refs.append(norm(d['free_form_answer']))
            if d.get('yes_no') is not None:refs.append('yes' if d['yes_no'] else 'no')
            if d.get('extractive_spans'):refs.append(norm(' '.join(d['extractive_spans'])))
        out.append({'paper_id':paper_id,'question_id':str(qa.get('question_id',f'{paper_id}_{qi}')),
                    'question':norm(qa.get('question')),'evidence':list(dict.fromkeys(norm(e) for e in ev if norm(e))),
                    'references':list(dict.fromkeys(refs)), 'unanswerable':unanswerable})
    return out


def gold_ids(paper,qa):
    # QASPER annotators provide the literal evidence paragraphs, sometimes unavailable
    # for figures/tables. Exact normalized matching avoids fabricated gold labels.
    lookup=defaultdict(set)
    for pid,_,txt in paragraphs(paper):lookup[norm(txt).casefold()].add(pid)
    mapped=set()
    for txt in qa['evidence']: mapped.update(lookup.get(norm(txt).casefold(),set()))
    return mapped


@dataclass(frozen=True)
class Config:
    chunking: str='paragraph'  # paragraph, fixed, recursive, semantic, sentence_window, parent, auto_merging
    retrieval: str='dense'     # dense, bm25, hybrid
    rerank: bool=False
    transformation: str='none' # none, rewrite, multi_query, hyde
    top_k: int=5
    candidate_k: int=15
    chunk_size: int=256
    overlap: int=32
    window_size: int=2
    merge_threshold: float=.5
    rrf_k: int=60
    embed_model: str=EMBED_NAME
    generator_model: str='qwen2.5:3b'
    semantic_buffer: int=1
    semantic_percentile: int=90
    parent_chunk_size: int=512
    child_chunk_size: int=128
    bm25_k1: float=1.5
    bm25_b: float=0.75
    dense_top_k: int=15
    sparse_top_k: int=15
    rerank_top_n: int=5
    reranker_model: str='cross-encoder/ms-marco-MiniLM-L-6-v2'
    query_count: int=3
    hyde_count: int=1


def build_nodes(paper_id,paper,cfg,embed):
    from llama_index.core import Document
    from llama_index.core.schema import TextNode
    from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser, SentenceWindowNodeParser
    source=[]; parents={}; children={}
    docs=[]
    for pid,section,txt in paragraphs(paper):
        metadata={'paper_id':paper_id, 'source_pid':pid,'section':section,'title':paper.get('title','')}
        docs.append(Document(text=txt, id_=f'{paper_id}::{pid}', metadata=metadata,
                             excluded_embed_metadata_keys=list(metadata), excluded_llm_metadata_keys=list(metadata)))
        parents[pid]=txt
    if cfg.chunking=='paragraph':
        nodes=[TextNode(id_=d.id_,text=d.text,metadata=d.metadata,
                        excluded_embed_metadata_keys=list(d.metadata),excluded_llm_metadata_keys=list(d.metadata)) for d in docs]
    elif cfg.chunking=='fixed':
        # TokenTextSplitter splits using token budget (not character count).
        from llama_index.core.node_parser import TokenTextSplitter
        parser=TokenTextSplitter(chunk_size=cfg.chunk_size,chunk_overlap=cfg.overlap)
        nodes=parser.get_nodes_from_documents(docs)
    elif cfg.chunking=='recursive':
        parser=SentenceSplitter(chunk_size=cfg.chunk_size,chunk_overlap=cfg.overlap)
        nodes=parser.get_nodes_from_documents(docs)
    elif cfg.chunking=='semantic':
        parser=SemanticSplitterNodeParser.from_defaults(embed_model=embed,buffer_size=cfg.semantic_buffer,breakpoint_percentile_threshold=cfg.semantic_percentile)
        nodes=parser.get_nodes_from_documents(docs)
    elif cfg.chunking=='sentence_window':
        parser=SentenceWindowNodeParser.from_defaults(window_size=cfg.window_size,window_metadata_key='window',original_text_metadata_key='original_text')
        nodes=parser.get_nodes_from_documents(docs)
    elif cfg.chunking in ('parent','auto_merging'):
        # Parent representation is source paragraph; children use configurable token window.
        # The parent size parameter controls eligible parent splitting when the source
        # paragraph exceeds that size. Ground truth remains original paragraph IDs.
        parent_parser=SentenceSplitter(chunk_size=cfg.parent_chunk_size,chunk_overlap=0)
        parent_nodes=parent_parser.get_nodes_from_documents(docs)
        child_parser=SentenceSplitter(chunk_size=cfg.child_chunk_size,chunk_overlap=min(cfg.overlap,cfg.child_chunk_size-1))
        nodes=[]
        for parent in parent_nodes:
            source_pid=parent.metadata['source_pid']
            parent_key=parent.node_id
            parents[parent_key]=parent.text
            for child in child_parser.get_nodes_from_documents([Document(text=parent.text,metadata={**parent.metadata,'parent_key':parent_key},id_=parent_key)]):
                child.metadata['source_pid']=source_pid
                child.metadata['parent_key']=parent_key
                nodes.append(child)
    else: raise ValueError(cfg.chunking)
    for n in nodes:
        pid=n.metadata.get('source_pid')
        children.setdefault(n.metadata.get('parent_key',pid),[]).append(n.node_id)
    return nodes,parents,children


def get_embedder(name=EMBED_NAME):
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    return HuggingFaceEmbedding(model_name=name)


class SearchLab:
    def __init__(self,pid,paper,cfg:Config, embed=None):
        from llama_index.core import VectorStoreIndex
        self.pid=pid;self.paper=paper;self.cfg=cfg
        self.embed=embed or get_embedder(cfg.embed_model)
        self.nodes,self.parents,self.children=build_nodes(pid,paper,cfg,self.embed)
        self.by_id={n.node_id:n for n in self.nodes}
        self.index=None;self.bm25=None;self.reranker=None
        if cfg.retrieval in ('dense','hybrid'):
            self.index=VectorStoreIndex(self.nodes,embed_model=self.embed,show_progress=False)
        if cfg.retrieval in ('bm25','hybrid'):
            from llama_index.retrievers.bm25 import BM25Retriever
            self.bm25=BM25Retriever.from_defaults(nodes=self.nodes,similarity_top_k=cfg.sparse_top_k)
        if cfg.rerank:
            from sentence_transformers import CrossEncoder
            self.reranker=CrossEncoder(cfg.reranker_model)

    def _query_variants(self,q):
        cfg=self.cfg
        if cfg.transformation=='none':return [q]
        prompt={
            'rewrite':f'Rewrite this academic research question for retrieving paper passages. Preserve meaning. Output only one query.\nQuestion: {q}',
            'multi_query':f'Write {cfg.query_count} diverse search queries for the research question, one per line, no numbering.\nQuestion: {q}',
            'hyde':f'Write a short hypothetical scientific-paper passage that could answer this question. Do not claim to have read the paper.\nQuestion: {q}'
        }[cfg.transformation]
        result=ollama_chat(prompt,cfg.generator_model)
        if cfg.transformation=='multi_query':
            # LLM output can contain numbered lines, empty bullets, or placeholders.
            # Keep the original question first and deduplicate cleaned variants.
            cleaned=[]
            for line in result.splitlines():
                candidate=re.sub(r'^\s*(?:[-*]\s*|\d+[.)]\s*)+', '', line).strip().strip('"`')
                if candidate and candidate.casefold() not in {'none', 'null', 'n/a'}:
                    cleaned.append(candidate)
            return list(dict.fromkeys([q]+cleaned[:cfg.query_count]))
        return [result.strip()] if result.strip() else [q]

    def search(self,question, top_k=None):
        from llama_index.core.schema import NodeWithScore
        k=top_k or self.cfg.top_k
        variants=self._query_variants(question)
        rankings=[]
        for variant_index, q in enumerate(variants):
            # Some versions of the BM25/dense stack raise TypeError when a
            # generated query produces an unusable internal score (None * float).
            # That query must not abort the entire paper/configuration. Always
            # require the original question to work, and explicitly report any
            # failed *additional* query, rather than silently inventing scores.
            if not isinstance(q, str) or not q.strip():
                continue
            sources=[]
            if self.index is not None:
                sources.append(('dense', lambda q=q: self.index.as_retriever(
                    similarity_top_k=self.cfg.dense_top_k).retrieve(q)))
            if self.bm25 is not None:
                sources.append(('bm25', lambda q=q: self.bm25.retrieve(q)))
            for source_name, retrieve in sources:
                try:
                    ranking=retrieve()
                except TypeError as exc:
                    if variant_index == 0 or 'NoneType' not in str(exc):
                        raise RuntimeError(
                            f'{source_name} retrieval failed for '
                            f'{"original" if variant_index == 0 else "generated"} query {q!r}'
                        ) from exc
                    import warnings
                    warnings.warn(
                        f'Skipping malformed multi-query {variant_index} ({source_name}): '
                        f'{exc}; question={q[:100]!r}', RuntimeWarning,
                        stacklevel=2,
                    )
                    continue
                if ranking:
                    rankings.append(ranking)
        if not rankings:
            raise RuntimeError(f'No successful retrieval results for question: {question[:120]!r}')
        if len(rankings)==1:
            candidates=rankings[0]
        else:
            # RRF rank fusion: mathematically comparable between BM25 and dense.
            scores=defaultdict(float);records={}
            for ranking in rankings:
                for i,result in enumerate(ranking,1):
                    nodeid=result.node.node_id
                    scores[nodeid]+=1/(self.cfg.rrf_k+i)
                    records[nodeid]=result.node
            candidates=[NodeWithScore(node=records[n],score=s) for n,s in sorted(scores.items(),key=lambda a:-a[1])]
        # Expansion after retrieval, preserving the source paragraph for evaluation.
        if self.cfg.chunking in ('parent','auto_merging'):
            retrieved_counts=Counter(x.node.metadata.get('parent_key',x.node.metadata.get('source_pid')) for x in candidates[:self.cfg.candidate_k])
            expanded=[];done=set()
            for x in candidates:
                pid=x.node.metadata.get('parent_key',x.node.metadata.get('source_pid'))
                can_expand=(self.cfg.chunking=='parent' or retrieved_counts[pid]/max(1,len(self.children.get(pid,[])))>=self.cfg.merge_threshold)
                if can_expand and pid in self.parents:
                    if pid in done:continue
                    done.add(pid)
                    from llama_index.core.schema import TextNode
                    n=TextNode(id_=f'{self.pid}::{pid}::expanded',text=self.parents[pid],metadata={**x.node.metadata,'expanded':True})
                    expanded.append(NodeWithScore(node=n,score=x.score))
                else:expanded.append(x)
            candidates=expanded
        if self.cfg.rerank and candidates:
            pairs=[(question,x.node.metadata.get('window',x.node.text) if self.cfg.chunking=='sentence_window' else x.node.text) for x in candidates]
            scores=self.reranker.predict(pairs)
            candidates=[v for _,v in sorted(zip(scores,candidates),key=lambda t:-float(t[0]))][:self.cfg.rerank_top_n]
        return candidates[:k]


def passage(hit,cfg):
    return hit.node.metadata.get('window',hit.node.text) if cfg.chunking=='sentence_window' else hit.node.text


def retrieval_metrics(hits,gold,k):
    # Paragraph-level de-duplication: duplicates from chunking should not inflate recall.
    ranked=[]
    for h in hits[:k]:
        sid=h.node.metadata.get('source_pid')
        if sid not in ranked:ranked.append(sid)
    rel=[int(x in gold) for x in ranked]
    found=sum(rel)
    precision=found/k
    recall=found/len(gold) if gold else float('nan')
    dcg=sum(v/math.log2(i+2) for i,v in enumerate(rel))
    ideal=sum(1/math.log2(i+2) for i in range(min(len(gold),k)))
    ap=sum(sum(rel[:i+1])/(i+1) for i,v in enumerate(rel) if v)/min(len(gold),k) if gold else float('nan')
    rr=next((1/(i+1) for i,v in enumerate(rel) if v),0)
    return {'precision_at_k':precision,'recall_at_k':recall,'hit_at_k':float(found>0),'mrr_at_k':rr,
            'map_at_k':ap,'ndcg_at_k':dcg/ideal if ideal else float('nan'),
            'retrieved_unique_paragraphs':len(ranked)}


def evaluate_retrieval(papers,cfg:Config,max_questions=50):
    import pandas as pd
    t0=time.perf_counter();rows=[];mapping={'questions_total':0,'questions_mappable':0,'questions_unmapped':0,'unanswerable':0}
    embed=get_embedder(cfg.embed_model)
    for pid,paper in papers.items():
        qas=annotations(pid,paper)
        chosen=[]
        for q in qas:
            if len(rows)+len(chosen)>=max_questions:break
            mapping['questions_total']+=1
            if q['unanswerable']:mapping['unanswerable']+=1
            gold=gold_ids(paper,q)
            if not gold: mapping['questions_unmapped']+=1;continue
            mapping['questions_mappable']+=1
            chosen.append((q,gold))
        if not chosen:continue
        lab=SearchLab(pid,paper,cfg,embed)
        for q,gold in chosen:
            start=time.perf_counter()
            hits=lab.search(q['question'])
            d=retrieval_metrics(hits,gold,cfg.top_k)
            rows.append({'paper_id':pid,'paper_title':paper.get('title',''),'question_id':q['question_id'],
                         'question':q['question'],'latency_s':time.perf_counter()-start,
                         'gold_count':len(gold),'retrieved_ids':[x.node.metadata.get('source_pid') for x in hits],
                         'gold_ids':sorted(gold),'contexts':[passage(x,cfg) for x in hits],
                         'references':q['references'],**d})
        if len(rows)>=max_questions:break
    mapping['runtime_s']=round(time.perf_counter()-t0,2)
    return pd.DataFrame(rows),mapping


def summary(df,cfg,mapping):
    metrics=['precision_at_k','recall_at_k','hit_at_k','mrr_at_k','map_at_k','ndcg_at_k','latency_s']
    return {**asdict(cfg),**{m:float(df[m].mean()) if len(df) else None for m in metrics},
            'evaluated_questions':len(df),**mapping}


def save_run(df,cfg,mapping,run_name):
    import pandas as pd
    ART.mkdir(exist_ok=True,parents=True)
    folder=ART/'runs';folder.mkdir(exist_ok=True)
    df.to_json(folder/f'{run_name}.json',orient='records',indent=2)
    (folder/f'{run_name}_summary.json').write_text(json.dumps(summary(df,cfg,mapping),indent=2))
    return folder/f'{run_name}_summary.json'


def ollama_chat(prompt,model='qwen2.5:3b',url='http://localhost:11434',json_mode=False):
    import requests
    r=requests.post(url+'/api/generate',json={'model':model,'prompt':prompt,'stream':False, **({'format':'json', 'options':{'temperature':0}} if json_mode else {})},timeout=180)
    r.raise_for_status()
    return r.json()['response']


def generate_answer(question, hits, cfg, max_chars=14000):
    context='\n\n'.join(f'[{i+1}] {passage(hit,cfg)}' for i,hit in enumerate(hits))[:max_chars]
    prompt=f'''Answer the research question using ONLY the evidence below. If insufficient, say you cannot determine it. Cite source numbers in square brackets.\nQUESTION: {question}\nEVIDENCE:\n{context}\nANSWER:'''
    return ollama_chat(prompt,cfg.generator_model)



def judge_scores(question, answer, contexts, model='qwen2.5:3b', max_attempts=3):
    """Local Ollama LLM-as-judge; return NaN for failed scores, never zero.

    These scores are exploratory proxies, not the same as RAGAS metrics.
    """
    import math
    prompt = f'''Evaluate this RAG response. Return a JSON object ONLY with numeric values 0 to 1:
{{"groundedness":0.0,"context_relevance":0.0,"answer_relevance":0.0}}
Groundedness: are the answer's factual claims supported by the retrieved context?
Context relevance: are the supplied passages relevant to the question?
Answer relevance: does the response address the question?
Question: {question}\nAnswer: {answer}\nContexts: {json.dumps(contexts,ensure_ascii=False)[:11000]}'''
    required = ('groundedness', 'context_relevance', 'answer_relevance')
    for attempt in range(1, max_attempts+1):
        try:
            txt = ollama_chat(prompt, model, json_mode=True)
            found = re.search(r'\{.*\}', txt, re.S)
            vals = json.loads(found.group(0)) if found else json.loads(txt)
            scores = {k: float(vals[k]) for k in required}
            if not all(math.isfinite(v) and 0 <= v <= 1 for v in scores.values()):
                raise ValueError('Judge scores must be finite values in [0,1]')
            return scores
        except Exception as exc:
            print(f'Local judge attempt {attempt}/{max_attempts} failed: {type(exc).__name__}: {str(exc)[:180]}')
            if attempt < max_attempts:
                time.sleep(min(attempt, 3))
    return {k: None for k in required}


def evaluate_generation(df,cfg,limit=8,with_judge=True):
    import pandas as pd
    records=[]
    for row in df.head(limit).to_dict('records'):
        # Reuse retrieval contexts from stored output; no reindex necessary.
        contexts=row['contexts']; question=row['question']
        from llama_index.core.schema import NodeWithScore,TextNode
        hits=[NodeWithScore(node=TextNode(text=t),score=1.) for t in contexts]
        answer=generate_answer(question,hits,cfg)
        scores=judge_scores(question,answer,contexts,cfg.generator_model) if with_judge else {}
        records.append({**row,'answer':answer,**scores})
    return pd.DataFrame(records)


def reference_token_f1(answer, references):
    """Deterministic best-over-references token F1 (not an LLM judge)."""
    import re
    from collections import Counter
    from ast import literal_eval
    if isinstance(references, str):
        try:
            parsed = literal_eval(references)
            references = parsed if isinstance(parsed, (list, tuple)) else [references]
        except (ValueError, SyntaxError):
            references = [references]
    if not isinstance(references, (list, tuple)):
        return float("nan")
    def toks(s):
        return re.findall(r"\w+", str(s).casefold())
    pred = toks(answer)
    scores = []
    for reference in references:
        gold = toks(reference)
        if not gold:
            continue
        common = sum((Counter(pred) & Counter(gold)).values())
        if not pred:
            scores.append(0.0)
            continue
        precision, recall = common / len(pred), common / len(gold)
        scores.append(2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return max(scores) if scores else float("nan")



async def run_ragas_optional(
    generation_df,
    model="qwen2.5:7b",
    *,
    models=None,
    max_attempts=2,
    retry_base_seconds=1.5,
    ollama_url="http://localhost:11434",
    embedding_model=None,
):
    """Evaluate cached QASPER answers with RAGAS and LOCAL Ollama judges.

    Defaults to qwen2.5:7b as judge, with optional ordered local-model
    fallbacks. Uses local sentence-transformer embeddings for AnswerRelevancy.
    No Gemini, OpenAI cloud API, or GOOGLE_API_KEY is involved.

    Keep this function async; run `await run_ragas_optional(...)` in Jupyter.
    """
    import ast as ast_module
    import asyncio
    import random
    import re
    import numpy as np
    import pandas as pd
    import requests
    from openai import AsyncOpenAI
    from ragas.llms import llm_factory
    from ragas.embeddings import HuggingFaceEmbeddings
    from ragas.metrics.collections import (
        Faithfulness, ContextPrecision, ContextRecall, AnswerRelevancy,
    )

    if models is None:
        models = [model]
    elif isinstance(models, str):
        models = [models]
    models = list(dict.fromkeys(m for m in models if m))
    if not models:
        raise ValueError("Specify at least one local Ollama judge model.")
    if max_attempts < 1:
        raise ValueError("max_attempts must be >= 1")

    required = {"question", "answer", "contexts", "references"}
    missing = required.difference(generation_df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}. Got {list(generation_df.columns)}")

    # Fail fast with an actionable message if Ollama is not running or a model is missing.
    base = ollama_url.rstrip("/")
    try:
        response = requests.get(f"{base}/api/tags", timeout=10)
        response.raise_for_status()
        installed = {m["name"] for m in response.json().get("models", [])}
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Ollama is not reachable at {base}. Open the Ollama app or run `ollama serve`."
        ) from exc
    available = [m for m in models if m in installed]
    if not available:
        raise RuntimeError(
            f"Judge model(s) {models} are not installed. Available: {sorted(installed)}. "
            f"Run `ollama pull {models[0]}` in Terminal."
        )
    if len(available) != len(models):
        print("Skipping missing local judge models:", sorted(set(models) - set(available)))
    models = available

    # Async client works inside Jupyter's already-running asyncio event loop.
    # No hidden provider retries: we own the metric-level retry loop below.
    client = AsyncOpenAI(
        api_key="ollama", base_url=f"{base}/v1/",
        timeout=180.0, max_retries=0,
    )
    # Same local embedding model used for the retrieval experiments by default.
    embeddings = HuggingFaceEmbeddings(model=embedding_model or EMBED_NAME)
    metric_cache = {}
    metric_names = ("faithfulness", "context_precision", "context_recall", "answer_relevancy")

    def metrics_for(model_id):
        if model_id not in metric_cache:
            llm = llm_factory(model_id, provider="openai", client=client)
            metric_cache[model_id] = {
                "faithfulness": Faithfulness(llm=llm),
                "context_precision": ContextPrecision(llm=llm),
                "context_recall": ContextRecall(llm=llm),
                "answer_relevancy": AnswerRelevancy(llm=llm, embeddings=embeddings),
            }
        return metric_cache[model_id]

    def as_list(value):
        if isinstance(value, (list, tuple, np.ndarray)):
            return [str(x) for x in value if x is not None and str(x).strip()]
        if isinstance(value, str) and value.strip():
            try:
                parsed = ast_module.literal_eval(value)
                if isinstance(parsed, (list, tuple)):
                    return [str(x) for x in parsed if x is not None and str(x).strip()]
            except (ValueError, SyntaxError):
                pass
            return [value]
        return []

    output = []
    try:
        for index, row in generation_df.iterrows():
            question = str(row["question"])
            answer = str(row["answer"])
            contexts = as_list(row["contexts"])
            references = as_list(row["references"])
            reference = references[0] if references else None
            record = {
                "question": question,
                "answer": answer,
                "reference": reference,
                "reference_count": len(references),
                "reference_token_f1": reference_token_f1(answer, references),
                "judge_model": None,
            }
            inputs = {
                "faithfulness": dict(user_input=question, response=answer, retrieved_contexts=contexts),
                "context_precision": dict(user_input=question, reference=reference, retrieved_contexts=contexts),
                "context_recall": dict(user_input=question, reference=reference, retrieved_contexts=contexts),
                "answer_relevancy": dict(user_input=question, response=answer),
            }
            for name in metric_names:
                record[name] = float("nan")
                record[name + "_error"] = ""
                record[name + "_model"] = None
                if not contexts or (name in ("context_precision", "context_recall") and not reference):
                    record[name + "_error"] = "Missing context or reference"
                    continue
                errors = []
                for model_id in models:
                    try:
                        metric = metrics_for(model_id)[name]
                    except Exception as exc:
                        errors.append(f"{model_id}: initialization: {type(exc).__name__}: {str(exc)[:180]}")
                        continue
                    for attempt in range(1, max_attempts + 1):
                        try:
                            score = await metric.ascore(**inputs[name])
                            value = float(score.value)
                            if not np.isfinite(value):
                                raise ValueError("Metric returned non-finite score")
                            record[name] = value
                            record[name + "_model"] = model_id
                            if record["judge_model"] is None:
                                record["judge_model"] = model_id
                            record[name + "_error"] = ""
                            break
                        except Exception as exc:
                            message = str(exc)
                            errors.append(f"{model_id} attempt {attempt}: {type(exc).__name__}: {message[:160]}")
                            # Local model failures are often structured-output problems.
                            if attempt < max_attempts:
                                delay = retry_base_seconds * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
                                print(f"Row {index}: {name} / {model_id} retrying in {delay:.1f}s: {message[:100]}")
                                await asyncio.sleep(delay)
                    if pd.notna(record[name]):
                        break
                    print(f"Row {index}: {name} failed with {model_id}; trying next local judge if configured")
                if pd.isna(record[name]):
                    record[name + "_error"] = " | ".join(errors)[-1200:]
                    print(f"Row {index}: {name} failed after local judge attempts")
            output.append(record)
    finally:
        await client.close()
    return pd.DataFrame(output)

# --- Compact, controlled experiment grids ----------------------------------
# Changing one family at a time makes results interpretable and avoids the
# full Cartesian-product explosion.
CHUNK_OPTIONS = {
    'paragraph': [{}],
    'fixed': [{'chunk_size':256,'overlap':32},
              {'chunk_size':512,'overlap':32}],
    'recursive': [{'chunk_size':256,'overlap':32},
                  {'chunk_size':512,'overlap':32}],
    'semantic': [{'semantic_buffer':1,'semantic_percentile':90},
                 {'semantic_buffer':2,'semantic_percentile':90}],
    'sentence_window': [{'window_size':1},{'window_size':3}],
    'parent': [{'parent_chunk_size':512,'child_chunk_size':128},
               {'parent_chunk_size':1024,'child_chunk_size':256}],
    'auto_merging': [{'parent_chunk_size':512,'child_chunk_size':128,'merge_threshold':0.35},
                     {'parent_chunk_size':512,'child_chunk_size':128,'merge_threshold':0.60}],
}
RETRIEVAL_OPTIONS = {
    'dense': [{'dense_top_k':10},{'dense_top_k':20}],
    'bm25': [{'sparse_top_k':10},{'sparse_top_k':20}],
    'hybrid': [{'dense_top_k':10,'sparse_top_k':10,'rrf_k':60},
               {'dense_top_k':20,'sparse_top_k':20,'rrf_k':30}],
}
RERANK_OPTIONS = [{'rerank':True,'rerank_top_n':5},
                  {'rerank':True,'rerank_top_n':10}]
TRANSFORM_OPTIONS = {
    'rewrite': [{}],
    'multi_query': [{'query_count':2},{'query_count':3}],
    'hyde': [{}]
}


def make_controlled_grid(base=None,chunk_options=None,retrieval_options=None,
                         rerank_options=None,transform_options=None,
                         include_llm=False):
    """Generate compact ablations (not a cross-product of all families).

    1. Baseline: paragraph + dense
    2. Chunk variants with dense retrieval
    3. Retriever variants with recursive-256 chunks
    4. Reranker variants with hybrid retrieval
    5. Optional LLM query transforms with the same hybrid setup
    Configs are deduplicated and ordered deterministically.
    """
    from dataclasses import replace
    base=base or Config(top_k=5,candidate_k=20)
    chunk_options=CHUNK_OPTIONS if chunk_options is None else chunk_options
    retrieval_options=RETRIEVAL_OPTIONS if retrieval_options is None else retrieval_options
    rerank_options=RERANK_OPTIONS if rerank_options is None else rerank_options
    transform_options=TRANSFORM_OPTIONS if transform_options is None else transform_options
    seen=set(); output=[]
    def add(cfg):
        if cfg.chunking in ('fixed','recursive') and cfg.overlap>=cfg.chunk_size:
            raise ValueError('Overlap must be smaller than chunk size')
        if cfg.rerank and cfg.rerank_top_n<cfg.top_k:
            raise ValueError('rerank_top_n must be >= top_k')
        key=run_id(cfg)
        if key not in seen:seen.add(key);output.append(cfg)
    baseline=replace(base,chunking='paragraph',retrieval='dense',rerank=False,transformation='none')
    add(baseline)
    for method,variations in chunk_options.items():
        for params in variations:
            add(replace(baseline,chunking=method,**params))
    retrieval_base=replace(base,chunking='recursive',chunk_size=256,overlap=32,
                            retrieval='dense',rerank=False,transformation='none')
    for method,variations in retrieval_options.items():
        for params in variations:
            add(replace(retrieval_base,retrieval=method,**params))
    hybrid_base=replace(retrieval_base,retrieval='hybrid',dense_top_k=10,
                        sparse_top_k=10,rrf_k=60)
    for params in rerank_options:
        add(replace(hybrid_base,**params))
    if include_llm:
        for method,variations in transform_options.items():
            for params in variations:
                add(replace(hybrid_base,transformation=method,**params))
    return output


def run_id(cfg):
    values=asdict(cfg)
    hashed=hashlib.sha256(json.dumps(values,sort_keys=True).encode()).hexdigest()[:12]
    return f"{cfg.chunking}_{cfg.retrieval}_{cfg.transformation}_{'rerank' if cfg.rerank else 'raw'}_{hashed}"


def run_sweep(papers,configs,max_questions=30,skip_completed=True,
              fail_fast=False,progress=None):
    """Run every selected configuration. NO time cutoff or max_runs limit."""
    import pandas as pd
    failures=[];records=[]
    configs=list(configs)
    folder=ART/'runs';folder.mkdir(exist_ok=True,parents=True)
    for i,cfg in enumerate(configs,1):
        name=run_id(cfg)
        path=folder/f'{name}_summary.json'
        if skip_completed and path.exists():
            item=json.loads(path.read_text())
            records.append({'run':name,'status':'cached',**item})
            if progress: progress(i,len(configs),name,'cached')
            continue
        try:
            df,mapping=evaluate_retrieval(papers,cfg,max_questions=max_questions)
            save_run(df,cfg,mapping,name)
            records.append({'run':name,'status':'ok',**summary(df,cfg,mapping)})
            if progress:progress(i,len(configs),name,'ok')
        except Exception as exc:
            import traceback
            failures.append({'run':name,'exception':repr(exc),
                             'traceback':traceback.format_exc(),**asdict(cfg)})
            if progress:progress(i,len(configs),name,f'error: {exc}')
            if fail_fast:raise
    pd.DataFrame(failures).to_json(ART/'sweep_failures.json',orient='records',indent=2)
    return pd.DataFrame(records),pd.DataFrame(failures)


Overwriting ragxiv_lab.py


In [4]:
import importlib, json, ragxiv_lab
importlib.reload(ragxiv_lab)
from ragxiv_lab import (
    Config, load_papers, annotations, gold_ids, paragraphs,
    evaluate_retrieval, summary, save_run, run_id,
    make_controlled_grid, run_sweep, evaluate_generation,
    run_ragas_optional
)


## 2. Load QASPER and validate evidence mapping

The official QASPER JSON has `full_text` as a list of section dictionaries; this notebook handles both that and column-oriented conversions. QASPER evidence is annotated at the paragraph level. We do **paper-scoped** retrieval and retain original paragraph IDs for consistent metrics.


In [5]:
N_PAPERS = 100
MAX_QUESTIONS = 300
papers = load_papers(split='dev', limit=N_PAPERS)
print('Loaded', len(papers), 'papers')
first_id, first_paper = next(iter(papers.items()))
print('Example:', first_paper.get('title', first_id))
print('Extracted paragraphs:', len(paragraphs(first_paper)))
first_qa = annotations(first_id, first_paper)
print('Questions:', len(first_qa))
if first_qa:
    print('Gold evidence paragraphs for first question:', len(gold_ids(first_paper, first_qa[0])))


Loaded 100 papers
Example: Cross-lingual Pre-training Based Transfer for Zero-shot Neural Machine Translation
Extracted paragraphs: 47
Questions: 4
Gold evidence paragraphs for first question: 1


In [6]:
baseline = Config(chunking='paragraph', retrieval='dense', rerank=False,
                  transformation='none', top_k=5, dense_top_k=10)
baseline_df, baseline_mapping = evaluate_retrieval(papers, baseline, max_questions=MAX_QUESTIONS)
save_run(baseline_df, baseline, baseline_mapping, run_id(baseline))
print(json.dumps(summary(baseline_df, baseline, baseline_mapping), indent=2))


/Users/aryan.agarwal/FlightPrediction/py_analysis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1124 10:26:51.497000 80946 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15943.05it/s]


{
  "chunking": "paragraph",
  "retrieval": "dense",
  "rerank": false,
  "transformation": "none",
  "top_k": 5,
  "candidate_k": 15,
  "chunk_size": 256,
  "overlap": 32,
  "window_size": 2,
  "merge_threshold": 0.5,
  "rrf_k": 60,
  "embed_model": "BAAI/bge-small-en-v1.5",
  "generator_model": "qwen2.5:3b",
  "semantic_buffer": 1,
  "semantic_percentile": 90,
  "parent_chunk_size": 512,
  "child_chunk_size": 128,
  "bm25_k1": 1.5,
  "bm25_b": 0.75,
  "dense_top_k": 10,
  "sparse_top_k": 15,
  "rerank_top_n": 5,
  "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
  "query_count": 3,
  "hyde_count": 1,
  "precision_at_k": 0.17549407114624505,
  "recall_at_k": 0.5238426221331054,
  "hit_at_k": 0.6956521739130435,
  "mrr_at_k": 0.44433465085638996,
  "map_at_k": 0.3307905138339921,
  "ndcg_at_k": 0.40956964781855654,
  "latency_s": 0.007441599324176257,
  "evaluated_questions": 253,
  "questions_total": 274,
  "questions_mappable": 253,
  "questions_unmapped": 21,
  "unanswerabl

## 3. Edit the compact experiment variations

Each method has **at most two variants**. Comparisons are one-factor-at-a-time rather than the Cartesian product of every chunker, retriever, reranker and transform.

- Chunking variants use dense retrieval.
- Retriever variants use recursive-256 chunking.
- Reranker variants use hybrid retrieval.
- Optional query transforms use the same hybrid setup and require Ollama.

Edit any parameter list below. Every selected configuration is executed—there is **no `MAX_RUNS` or runtime cutoff**.


In [7]:
from ragxiv_lab import CHUNK_OPTIONS, RETRIEVAL_OPTIONS, RERANK_OPTIONS, TRANSFORM_OPTIONS

# Method -> list of parameter dictionaries (max 2 per method by default).
CHUNK_GRID = {
    'paragraph': [{}],
    'fixed': [{'chunk_size': 256, 'overlap': 32},
              {'chunk_size': 512, 'overlap': 32}],
    'recursive': [{'chunk_size': 256, 'overlap': 32},
                  {'chunk_size': 512, 'overlap': 32}],
    'semantic': [{'semantic_buffer': 1, 'semantic_percentile': 90},
                 {'semantic_buffer': 2, 'semantic_percentile': 90}],
    'sentence_window': [{'window_size': 1}, {'window_size': 3}],
    'parent': [{'parent_chunk_size': 512, 'child_chunk_size': 256},
               {'parent_chunk_size': 1024, 'child_chunk_size': 256}],
    'auto_merging': [{'parent_chunk_size': 512, 'child_chunk_size': 256, 'merge_threshold': 0.35},
                     {'parent_chunk_size': 512, 'child_chunk_size': 256, 'merge_threshold': 0.60}],
}
RETRIEVAL_GRID = {
    'dense': [{'dense_top_k': 10}, {'dense_top_k': 20}],
    'bm25': [{'sparse_top_k': 10}, {'sparse_top_k': 20}],
    'hybrid': [{'dense_top_k': 10, 'sparse_top_k': 10, 'rrf_k': 60},
               {'dense_top_k': 20, 'sparse_top_k': 20, 'rrf_k': 30}],
}
RERANK_GRID = [
    {'rerank': True, 'rerank_top_n': 5},
    {'rerank': True, 'rerank_top_n': 10},
]
TRANSFORM_GRID = {
    'rewrite': [{}],
    'multi_query': [{'query_count': 2}, {'query_count': 3}],
    'hyde': [{}],
}
BASE = Config(top_k=5, candidate_k=20, dense_top_k=10, sparse_top_k=10)
INCLUDE_LLM_TRANSFORMS = True  # True requires local Ollama
SKIP_COMPLETED = True
STOP_ON_ERROR = False


## 4. Preview all selected configurations

Each row is a full configuration, and duplicates are removed. No random sampling, slicing, time limit or run cap is applied.


In [8]:
from dataclasses import asdict
import pandas as pd

configs = make_controlled_grid(
    base=BASE,
    chunk_options=CHUNK_GRID,
    retrieval_options=RETRIEVAL_GRID,
    rerank_options=RERANK_GRID,
    transform_options=TRANSFORM_GRID,
    include_llm=INCLUDE_LLM_TRANSFORMS,
)
print('Total unique experiments:', len(configs))
preview = pd.DataFrame([{'run':run_id(c),**asdict(c)} for c in configs])
display(preview[['run','chunking','retrieval','rerank','transformation',
                 'chunk_size','overlap','window_size','semantic_percentile',
                 'rerank_top_n']])


Total unique experiments: 24


,run,chunking,retrieval,rerank,transformation,chunk_size,overlap,window_size,semantic_percentile,rerank_top_n
0,paragraph_dense_none_raw_ccd47dc3f09c,paragraph,dense,False,none,256,32,2,90,5
1,fixed_dense_none_raw_bd82b0a1f44f,fixed,dense,False,none,256,32,2,90,5
2,fixed_dense_none_raw_33d59b9de9aa,fixed,dense,False,none,512,32,2,90,5
3,recursive_dense_none_raw_07bf33c4b819,recursive,dense,False,none,256,32,2,90,5
4,recursive_dense_none_raw_97fdd06a2ea1,recursive,dense,False,none,512,32,2,90,5
5,semantic_dense_none_raw_1bfe3ed3280e,semantic,dense,False,none,256,32,2,90,5
6,semantic_dense_none_raw_c140ffeb308b,semantic,dense,False,none,256,32,2,90,5
7,sentence_window_dense_none_raw_b5e3fa5506ac,sentence_window,dense,False,none,256,32,1,90,5
8,sentence_window_dense_none_raw_18ee611b8977,sentence_window,dense,False,none,256,32,3,90,5
9,parent_dense_none_raw_e3ee3bc5f48b,parent,dense,False,none,256,32,2,90,5


## 5. Execute ALL selected experiments and save results

Completed retrieval runs are reused when `SKIP_COMPLETED=True`. The multi-query path now cleans malformed Ollama-generated queries; if an additional generated query causes the known `NoneType * float` retrieval error, only that query-source result is skipped with a visible warning. A failure on the original question is **not** hidden. The sweep also saves full tracebacks to `artifacts/sweep_failures.json` for any remaining failures.


In [9]:
def report_progress(i,total,name,status):
    print(f'[{i}/{total}] {name}: {status}',flush=True)

sweep_results,sweep_failures=run_sweep(
    papers, configs,
    max_questions=MAX_QUESTIONS,
    skip_completed=SKIP_COMPLETED,
    fail_fast=STOP_ON_ERROR,
    progress=report_progress,
)
print(f'Completed/cached: {len(sweep_results)} | failed: {len(sweep_failures)}')
if len(sweep_results):
    display(sweep_results.sort_values('recall_at_k',ascending=False)[
        ['chunking','retrieval','rerank','recall_at_k','precision_at_k',
         'mrr_at_k','ndcg_at_k','latency_s','evaluated_questions']].head(25))
if len(sweep_failures):display(sweep_failures[['run','exception']])


[1/24] paragraph_dense_none_raw_ccd47dc3f09c: cached
[2/24] fixed_dense_none_raw_bd82b0a1f44f: cached
[3/24] fixed_dense_none_raw_33d59b9de9aa: cached
[4/24] recursive_dense_none_raw_07bf33c4b819: cached
[5/24] recursive_dense_none_raw_97fdd06a2ea1: cached
[6/24] semantic_dense_none_raw_1bfe3ed3280e: cached
[7/24] semantic_dense_none_raw_c140ffeb308b: cached
[8/24] sentence_window_dense_none_raw_b5e3fa5506ac: cached
[9/24] sentence_window_dense_none_raw_18ee611b8977: cached
[10/24] parent_dense_none_raw_e3ee3bc5f48b: cached
[11/24] parent_dense_none_raw_9e7c48d53d78: cached
[12/24] auto_merging_dense_none_raw_fecba2351f90: cached
[13/24] auto_merging_dense_none_raw_e9de941cb339: cached
[14/24] recursive_dense_none_raw_04818043eb3f: cached
[15/24] recursive_bm25_none_raw_c1afb95b5778: cached
[16/24] recursive_bm25_none_raw_51a2f48b860e: cached
[17/24] recursive_hybrid_none_raw_3f1778529753: cached
[18/24] recursive_hybrid_none_raw_579ce42d57d7: cached
[19/24] recursive_hybrid_none_reran

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20266.77it/s]


[22/24] recursive_hybrid_multi_query_raw_3804b7abb907: ok


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8249.40it/s]


[23/24] recursive_hybrid_multi_query_raw_3a057d3da38c: ok


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8466.55it/s]


[24/24] recursive_hybrid_hyde_raw_70b1e8d137e3: ok
Completed/cached: 24 | failed: 0


,chunking,retrieval,rerank,recall_at_k,precision_at_k,mrr_at_k,ndcg_at_k,latency_s,evaluated_questions
19,recursive,hybrid,True,0.536512,0.180237,0.485046,0.441213,0.076421,253
18,recursive,hybrid,True,0.536512,0.180237,0.485046,0.441213,0.065016,253
17,recursive,hybrid,False,0.528316,0.177075,0.461462,0.419190,0.007496,253
2,fixed,dense,False,0.525819,0.176285,0.448287,0.411593,0.006261,253
3,recursive,dense,False,0.524831,0.175494,0.444532,0.410723,0.006259,253
13,recursive,dense,False,0.524831,0.175494,0.444532,0.410723,0.007600,253
0,paragraph,dense,False,0.523843,0.175494,0.444335,0.409570,0.006837,253
4,recursive,dense,False,0.523843,0.175494,0.446311,0.410064,0.006905,253
16,recursive,hybrid,False,0.518958,0.173913,0.456456,0.413494,0.007709,253
1,fixed,dense,False,0.517123,0.173123,0.441304,0.404766,0.006581,253


## 6. Optional generated-answer evaluation and RAGAS

The retrieval results do not need an LLM. Generation uses local Ollama. The optional RAGAS 0.4 **collections** evaluation uses Ollama via local's OpenAI-compatible endpoint (no OpenAI API key), and a local embedding model for answer relevancy. Set `No cloud key` via `~/RAGxiv/.env` and load it in the next cell. Results are written to `artifacts/ragas_scores.csv` and displayed in Streamlit. For QASPER multiple reference answers, the first nonempty reference is used for reference-based metrics; unanswerable/no-reference cases have unavailable reference metrics. API quotas and charges depend on your Ollama tier.

**If you edit the engine:** rerun the `%%writefile ragxiv_lab.py` cell and the import/reload cell before executing this section. The RAGAS function is async and must be called with `await` in Jupyter.
**Updated:** Ollama API failures (429/503) are retried per metric, and failed scores are recorded separately. Reference token-F1 directly compares each Ollama answer with QASPER human answers without an LLM call.


Ollama judge evaluation tries the priority-ordered `GEMINI_JUDGE_MODELS` list. Each RAGAS metric records its actual successful model in a `*_model` column. Missing local Ollama judge scores are reevaluated from cached answers; no reretrieval is needed.

**Local judge:** `qwen2.5:7b` via Ollama; answer generation remains `qwen2.5:3b`. Run `ollama pull qwen2.5:7b` once. RAGAS AnswerRelevancy uses local Hugging Face embeddings (first run may download model weights). No Gemini API key or web-search API is used. The judge may fail to produce valid structured output for some questions; failed scores remain `NaN` with diagnostic columns.


In [10]:
# Local RAGAS judge preflight (no .env or Gemini key needed)
import requests
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_RAGAS_MODEL = "qwen2.5:7b"
try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
    response.raise_for_status()
    available_models = [m["name"] for m in response.json().get("models", [])]
    print("Ollama models:", available_models)
    if OLLAMA_RAGAS_MODEL not in available_models:
        print(f"Required judge missing: run `ollama pull {OLLAMA_RAGAS_MODEL}` in Terminal")
except requests.RequestException as exc:
    print("Ollama is not running:", exc)
    print("Launch the Ollama macOS app before running the generation or judge steps.")


Ollama models: ['qwen2.5:7b', 'qwen2.5:3b']


In [11]:
RUN_GENERATION = True
RUN_RAGAS = True
EVAL_GENERATION_LIMIT = 30
OLLAMA_JUDGE_MODELS = ["qwen2.5:7b"]  # locally installed Ollama judge
REPAIR_LOCAL_JUDGE_SCORES = True
LOCAL_JUDGE_MODEL = "qwen2.5:7b"  # judge differs from answer generator (qwen2.5:3b)

from pathlib import Path
from dataclasses import fields
import pandas as pd
import ast

saved_generation = Path("artifacts/generation_scores.csv")
if RUN_GENERATION:
    if saved_generation.exists():
        generation_scores = pd.read_csv(saved_generation)
        print("Using cached Ollama answers; no retrieval or generation rerun.")
    else:
        if sweep_results.empty:
            raise RuntimeError("Run retrieval experiments before generating answers.")
        winner_id = sweep_results.sort_values("recall_at_k", ascending=False).iloc[0]["run"]
        winner_row = sweep_results.loc[sweep_results["run"] == winner_id].iloc[0]
        winner_cfg = Config(**{
            f.name: winner_row[f.name] for f in fields(Config)
            if f.name in winner_row and pd.notna(winner_row[f.name])
        })
        winner_data = pd.read_json(f"artifacts/runs/{winner_id}.json")
        generation_scores = evaluate_generation(winner_data, winner_cfg, limit=EVAL_GENERATION_LIMIT)
        saved_generation.parent.mkdir(parents=True, exist_ok=True)
        generation_scores.to_csv(saved_generation, index=False)

# Reuse saved answers even when RUN_GENERATION=False.
if (RUN_RAGAS or REPAIR_LOCAL_JUDGE_SCORES) and "generation_scores" not in globals():
    if not saved_generation.exists():
        raise FileNotFoundError("No cached answers found. Enable RUN_GENERATION once.")
    generation_scores = pd.read_csv(saved_generation)

# Rejudge cached answers if local scores were missing; do not regenerate answers.
if REPAIR_LOCAL_JUDGE_SCORES:
    import math
    from ragxiv_lab import judge_scores
    judge_cols = ["groundedness", "context_relevance", "answer_relevance"]
    for col in judge_cols:
        if col not in generation_scores:
            generation_scores[col] = float("nan")
    for idx, row in generation_scores.iterrows():
        if generation_scores.loc[idx, judge_cols].notna().all():
            continue
        contexts = row["contexts"]
        if isinstance(contexts, str):
            try:
                contexts = ast.literal_eval(contexts)
            except (ValueError, SyntaxError):
                contexts = [contexts]
        if not isinstance(contexts, list):
            contexts = []
        scores = judge_scores(
            str(row["question"]), str(row["answer"]), contexts,
            model=LOCAL_JUDGE_MODEL, max_attempts=3,
        )
        for col in judge_cols:
            if scores.get(col) is not None:
                generation_scores.at[idx, col] = scores[col]
        # Persist successful results immediately in case evaluation is interrupted.
        generation_scores.to_csv(saved_generation, index=False)
    print("Local judge successful scores:", generation_scores[judge_cols].notna().sum().to_dict())
display(generation_scores[[c for c in ["question", "answer", "groundedness", "context_relevance", "answer_relevance"] if c in generation_scores]].head())

if RUN_RAGAS:
    # RAGAS uses cached answers even if RUN_GENERATION=False.
    if "generation_scores" not in globals():
        if not saved_generation.exists():
            raise FileNotFoundError("No saved generation scores; run generation once first.")
        generation_scores = pd.read_csv(saved_generation)
    ragas_scores = await run_ragas_optional(
        generation_scores, models=OLLAMA_JUDGE_MODELS, max_attempts=2
    )
    ragas_scores.to_csv("artifacts/ragas_scores.csv", index=False)
    cols = ["reference_token_f1", "faithfulness", "context_precision", "context_recall", "answer_relevancy"]
    display(ragas_scores[[c for c in cols if c in ragas_scores]].head())
    error_cols = [c for c in ragas_scores if c.endswith("_error")]
    print("Success counts:", ragas_scores[[c for c in cols if c in ragas_scores]].notna().sum().to_dict())
    print("Metric error counts:", {c: int(ragas_scores[c].fillna("").ne("").sum()) for c in error_cols})


Local judge successful scores: {'groundedness': 30, 'context_relevance': 30, 'answer_relevance': 30}


,question,answer,groundedness,context_relevance,answer_relevance
0,which multilingual approaches do they compare ...,The evidence provided compares the proposed ap...,0.8,0.9,0.9
1,what are the pivot-based baselines?,"Based on the evidence provided, pivot-based ba...",0.0,0.0,0.0
2,which datasets did they experiment with?,"Based on the evidence provided, the datasets t...",0.8,0.9,1.0
3,what language pairs are explored?,"Based on the provided evidence, the language p...",0.8,0.9,0.9
4,what ner models were evaluated?,The research evaluated three ner models: Stanf...,0.0,0.0,0.0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9106.21it/s]


Row 10: faithfulness / qwen2.5:7b retrying in 1.6s: The output is incomplete due to a max_tokens length limit.
Row 10: faithfulness failed with qwen2.5:7b; trying next local judge if configured
Row 10: faithfulness failed after local judge attempts
Row 14: faithfulness / qwen2.5:7b retrying in 1.7s: The output is incomplete due to a max_tokens length limit.
Row 14: faithfulness failed with qwen2.5:7b; trying next local judge if configured
Row 14: faithfulness failed after local judge attempts
Row 19: faithfulness / qwen2.5:7b retrying in 1.5s: The output is incomplete due to a max_tokens length limit.
Row 19: faithfulness failed with qwen2.5:7b; trying next local judge if configured
Row 19: faithfulness failed after local judge attempts


,reference_token_f1,faithfulness,context_precision,context_recall,answer_relevancy
0,0.037267,1.0,0.000000,0.000000,0.869796
1,0.083333,1.0,1.000000,1.000000,0.795326
2,0.190476,0.5,0.583333,1.000000,0.754400
3,0.358209,0.8,0.450000,0.333333,0.831610
4,0.246154,1.0,1.000000,1.000000,0.670723


Success counts: {'reference_token_f1': 30, 'faithfulness': 27, 'context_precision': 30, 'context_recall': 30, 'answer_relevancy': 30}
Metric error counts: {'faithfulness_error': 3, 'context_precision_error': 0, 'context_recall_error': 0, 'answer_relevancy_error': 0}


In [12]:
print("Available generation columns:", generation_scores.columns.tolist())
if "references" in generation_scores:
    display(generation_scores[["question", "references"]].head())

Available generation columns: ['paper_id', 'paper_title', 'question_id', 'question', 'latency_s', 'gold_count', 'retrieved_ids', 'gold_ids', 'contexts', 'references', 'precision_at_k', 'recall_at_k', 'hit_at_k', 'mrr_at_k', 'map_at_k', 'ndcg_at_k', 'retrieved_unique_paragraphs', 'answer', 'groundedness', 'context_relevance', 'answer_relevance']


,question,references
0,which multilingual approaches do they compare ...,"[BIBREF19 BIBREF20, multilingual NMT (MNMT) BI..."
1,what are the pivot-based baselines?,"[pivoting pivoting$_{\rm m}$, firstly translat..."
2,which datasets did they experiment with?,"[Europarl MultiUN, Europarl BIBREF31 MultiUN B..."
3,what language pairs are explored?,"[De-En, En-Fr, Fr-En, En-Es, Ro-En, En-De, Ar-..."
4,what ner models were evaluated?,[Stanford NER spaCy 2.0 recurrent model with a...


## 7. Generate Streamlit dashboard with visual diagnostics

The `%%writefile app.py` cell below builds the Streamlit dashboard from within the notebook. It includes a leaderboard, metric charts, latency/accuracy Pareto frontier, parameter sensitivity and chunking/retrieval heatmap, per-question score distributions, outliers and hardest examples, plus optional LLM-judge and RAGAS charts.

**These charts use real saved experiment results only**, and will be empty before the corresponding runs finish. Edit the notebook cell and rerun it to update the dashboard.


In [13]:
%%writefile app.py
"""This file is GENERATED by the RAGxiv notebook. Edit the %%writefile cell."""
import json
import time
from pathlib import Path
import pandas as pd
import altair as alt
import streamlit as st
from ragxiv_lab import load_papers, Config, SearchLab, get_embedder, generate_answer, passage

st.set_page_config(page_title='RAGxiv — Compact Experiment Lab', page_icon='📚',layout='wide')
st.title('📚 RAGxiv — Compact Experiment Lab')
st.caption('LlamaIndex | QASPER | paper-scoped, evidence-grounded retrieval evaluations')
ART=Path('artifacts')
qa,eval_tab=st.tabs(['🔎 Ask a paper','📊 Parameter sweep dashboard'])

@st.cache_data(show_spinner='Loading papers...')
def papers_cached(n):return load_papers('dev',n)

@st.cache_resource(show_spinner='Loading embeddings...')
def embed_cached(model):return get_embedder(model)

@st.cache_resource(show_spinner='Indexing selected paper...')
def search_cached(pid,cfg_json):
    cfg=Config(**json.loads(cfg_json))
    papers=papers_cached(12)
    return SearchLab(pid,papers[pid],cfg,embed_cached(cfg.embed_model))

run_folder=ART/'runs'
@st.cache_data(ttl=10)
def fetch_runs():
    rows=[]
    for f in sorted(run_folder.glob('*_summary.json')) if run_folder.exists() else []:
        try:rows.append({'run':f.name.removesuffix('_summary.json'),**json.loads(f.read_text())})
        except (OSError,ValueError):pass
    return pd.DataFrame(rows)

def pareto_frontier(frame, quality, cost):
    """Configurations not dominated on higher quality and lower latency."""
    rows = frame[[quality, cost]].apply(pd.to_numeric, errors='coerce')
    valid = rows.notna().all(axis=1) & (rows[cost] >= 0)
    frontier = pd.Series(False, index=frame.index)
    for idx in rows.index[valid]:
        q, t = rows.loc[idx, quality], rows.loc[idx, cost]
        others = rows.loc[valid & (rows.index != idx)]
        dominated = ((others[quality] >= q) & (others[cost] <= t) &
                     ((others[quality] > q) | (others[cost] < t))).any()
        frontier.loc[idx] = not dominated
    return frontier


def run_questions(run):
    p = run_folder / f'{run}.json'
    if not p.exists():
        return pd.DataFrame()
    try:
        return pd.read_json(p)
    except (OSError, ValueError):
        return pd.DataFrame()


with eval_tab:
    st.subheader('Experiment leaderboard & diagnostics')
    df = fetch_runs()
    if df.empty:
        st.info('No experiments yet. Run the sweep cells in the notebook, then refresh.')
    else:
        selected = df.copy()
        filter_cols = st.columns(4)
        for col, key in zip(filter_cols, ['chunking','retrieval','transformation','rerank']):
            if key in df:
                options = sorted(df[key].dropna().unique().tolist(), key=str)
                chosen = col.multiselect(key, options, default=options, format_func=str)
                selected = selected[selected[key].isin(chosen)]

        nums = [k for k in [
            'chunk_size','overlap','semantic_buffer','semantic_percentile','window_size',
            'parent_chunk_size','child_chunk_size','merge_threshold','dense_top_k','sparse_top_k',
            'rrf_k','candidate_k','rerank_top_n','query_count'
        ] if k in selected.columns]
        with st.expander('Filter individual hyperparameters'):
            choice = st.selectbox('Parameter', nums) if nums else None
            if choice:
                values = sorted(selected[choice].dropna().unique().tolist())
                picked = st.multiselect('Include values', values, default=values)
                selected = selected[selected[choice].isin(picked)]

        metrics = [m for m in ['recall_at_k','precision_at_k','mrr_at_k','map_at_k',
                                'ndcg_at_k','hit_at_k','latency_s'] if m in selected.columns]
        if selected.empty or not metrics:
            st.warning('No matching experiments or numeric metrics.')
        else:
            for m in metrics:
                selected[m] = pd.to_numeric(selected[m], errors='coerce')
            a, b, c = st.columns([2,1,1])
            metric = a.selectbox('Primary accuracy metric', [m for m in metrics if m != 'latency_s'] or metrics,
                                 index=0)
            sort_ascending = b.checkbox('Lowest first', value=False)
            max_display = c.number_input('Display top N', min_value=5, max_value=500, value=30, step=5)
            selected = selected.sort_values(metric, ascending=sort_ascending, na_position='last').copy()
            st.caption(f'{len(selected)} matching experiments. All figures use saved experiment results; no simulated scores.')

            # Warning: mean metrics are directly comparable only with the same evaluated questions.
            for field in ['top_k','evaluated_questions','questions_mappable']:
                if field in selected and selected[field].nunique(dropna=True) > 1:
                    st.warning(f'Runs differ in {field}. Interpret accuracy comparisons with care.')

            tab_overview, tab_trades, tab_sensitivity, tab_failures = st.tabs([
                '🏆 Performance', '⚖️ Accuracy vs speed', '🎛️ Parameter sensitivity', '🔍 Outliers & failures'
            ])

            with tab_overview:
                st.markdown('#### Standout configurations')
                ranked = selected.dropna(subset=[metric]).copy()
                if len(ranked):
                    best = ranked.nlargest(1, metric).iloc[0]
                    low_lat = (selected.dropna(subset=['latency_s']).nsmallest(1,'latency_s').iloc[0]
                               if 'latency_s' in selected and selected['latency_s'].notna().any() else None)
                    base = selected[(selected.get('chunking')=='paragraph') &
                                    (selected.get('retrieval')=='dense') &
                                    (selected.get('transformation')=='none') &
                                    (selected.get('rerank')==False)]
                    cards = st.columns(3)
                    cards[0].metric('Best '+metric, f'{best[metric]:.3f}', help=str(best['run']))
                    if low_lat is not None:
                        cards[1].metric('Fastest query latency', f'{low_lat["latency_s"]:.3f} s',
                                        help=str(low_lat['run']))
                    if not base.empty and pd.notna(base.iloc[0][metric]):
                        cards[2].metric('Best vs paragraph baseline', f'{best[metric]-base.iloc[0][metric]:+.3f}',
                                        help='Absolute difference, not a causal estimate')
                    else:
                        cards[2].metric('Evaluated configurations', len(selected))

                    st.markdown('#### Top configurations')
                    top = ranked.head(max_display).copy()
                    top['label'] = top['run'].astype(str)
                    chart = alt.Chart(top).mark_bar().encode(
                        y=alt.Y('label:N', sort='-x', title='Experiment'),
                        x=alt.X(f'{metric}:Q', title=metric, scale=alt.Scale(zero=True)),
                        color=alt.Color('retrieval:N', legend=alt.Legend(title='Retriever')),
                        tooltip=['run:N','chunking:N','retrieval:N',alt.Tooltip(f'{metric}:Q',format='.3f'),
                                 alt.Tooltip('latency_s:Q',format='.3f')]
                    ).properties(height=max(280, min(800, len(top)*22)))
                    st.altair_chart(chart, use_container_width=True)

                st.markdown('#### All saved results')
                display = ['run','chunking','retrieval','transformation','rerank','top_k',
                           'chunk_size','overlap','semantic_buffer','semantic_percentile','window_size',
                           'merge_threshold','dense_top_k','sparse_top_k','rrf_k','rerank_top_n',
                           'evaluated_questions',*metrics]
                st.dataframe(selected[[x for x in display if x in selected]],hide_index=True,use_container_width=True)
                st.download_button('Export filtered runs as CSV', selected.to_csv(index=False).encode(),
                                   'ragxiv_filtered_results.csv','text/csv')

            with tab_trades:
                st.markdown('#### Accuracy–latency Pareto frontier')
                st.caption('A point is Pareto-optimal if no other configuration has both at least as good '
                           'accuracy and at most as high latency, with one strict improvement.')
                if 'latency_s' in selected and selected['latency_s'].notna().sum() >= 2:
                    trade = selected.dropna(subset=[metric,'latency_s']).copy()
                    trade['frontier'] = pareto_frontier(trade, metric, 'latency_s').map({True:'Pareto',False:'Dominated'})
                    trade['label'] = trade['run'].astype(str)
                    scat = alt.Chart(trade).mark_circle(size=110,opacity=.85).encode(
                        x=alt.X('latency_s:Q',title='Average query latency (seconds)'),
                        y=alt.Y(f'{metric}:Q',title=metric,scale=alt.Scale(zero=False)),
                        color=alt.Color('frontier:N',scale=alt.Scale(domain=['Pareto','Dominated'],range=['#0d9488','#94a3b8'])),
                        shape='retrieval:N',
                        tooltip=['run:N','chunking:N','retrieval:N',alt.Tooltip(f'{metric}:Q',format='.3f'),
                                 alt.Tooltip('latency_s:Q',format='.3f')]
                    ).interactive().properties(height=390)
                    st.altair_chart(scat,use_container_width=True)
                    st.markdown('#### Non-dominated configurations')
                    st.dataframe(trade[trade['frontier']=='Pareto'][['run',metric,'latency_s','chunking','retrieval']]
                                 .sort_values(metric,ascending=False),hide_index=True,use_container_width=True)
                else:
                    st.info('At least two runs with latency measurements are required.')

                if 'retrieval' in selected and selected['retrieval'].nunique()>1:
                    st.markdown('#### Retrieval method comparison')
                    grouped=selected.groupby('retrieval',as_index=False).agg(
                        avg_metric=(metric,'mean'),n_runs=('run','count'))
                    st.altair_chart(alt.Chart(grouped).mark_bar().encode(
                        x='retrieval:N',y=alt.Y('avg_metric:Q',title='Average '+metric),
                        tooltip=['retrieval:N','n_runs:Q','avg_metric:Q']
                    ),use_container_width=True)
                    st.caption('Descriptive averages only: different retrieval groups may have different chunking settings.')

            with tab_sensitivity:
                st.markdown('#### Hyperparameter sensitivity')
                varying = [x for x in nums if selected[x].nunique(dropna=True)>1]
                if varying:
                    param = st.selectbox('Parameter to plot', varying)
                    pdat=selected[[param,metric,'run','chunking','retrieval']].dropna(subset=[param,metric]).copy()
                    pdat[param]=pdat[param].astype(str)
                    points=alt.Chart(pdat).mark_circle(size=85,opacity=.65).encode(
                        x=alt.X(f'{param}:N',title=param),y=alt.Y(f'{metric}:Q',scale=alt.Scale(zero=False)),
                        color='chunking:N',tooltip=['run:N',f'{param}:N',f'{metric}:Q','retrieval:N']
                    )
                    means=alt.Chart(pdat).mark_tick(color='black',size=32,thickness=3).encode(
                        x=alt.X(f'{param}:N'), y=f'mean({metric}):Q'
                    )
                    st.altair_chart((points+means).properties(height=320),use_container_width=True)
                    st.caption('Dots are individual runs; dark ticks are group means. This is NOT a controlled '
                               'effect estimate if other settings vary between runs.')
                else:
                    st.info('Run at least two values of a parameter to see a sensitivity plot.')

                if {'chunking','retrieval',metric}.issubset(selected.columns):
                    st.markdown('#### Chunking × retrieval heatmap')
                    heat=selected.groupby(['chunking','retrieval'],as_index=False).agg(
                        mean_score=(metric,'mean'),n_runs=('run','count'))
                    heatmap=alt.Chart(heat).mark_rect().encode(
                        x='retrieval:N',y='chunking:N',
                        color=alt.Color('mean_score:Q',title='Mean '+metric,scale=alt.Scale(scheme='tealblues')),
                        tooltip=['chunking:N','retrieval:N','n_runs:Q',alt.Tooltip('mean_score:Q',format='.3f')]
                    ).properties(height=280)
                    st.altair_chart(heatmap,use_container_width=True)
                    st.caption('Empty combinations were not tested. Cells average matching configurations only.')

            with tab_failures:
                st.markdown('#### Per-question distributions and hard cases')
                runs = selected['run'].tolist()
                chosen_run = st.selectbox('Inspect experiment',runs,key='inspect_run')
                questions = run_questions(chosen_run)
                if questions.empty:
                    st.warning('Per-question results unavailable for this run.')
                else:
                    qmetrics=[m for m in metrics if m in questions]
                    if qmetrics:
                        qm=st.selectbox('Question-level metric',qmetrics,index=qmetrics.index(metric) if metric in qmetrics else 0)
                        qdist=questions[[qm]].copy()
                        qdist[qm]=pd.to_numeric(qdist[qm],errors='coerce')
                        histogram=alt.Chart(qdist.dropna()).mark_bar().encode(
                            x=alt.X(f'{qm}:Q',bin=alt.Bin(maxbins=16),title=qm),
                            y=alt.Y('count():Q',title='Number of questions')
                        ).properties(height=220)
                        st.altair_chart(histogram,use_container_width=True)
                        if qm!='latency_s':
                            hard=questions.sort_values(qm,ascending=True).head(10)
                            st.markdown('**Lowest-scoring questions**')
                        else:
                            hard=questions.sort_values(qm,ascending=False).head(10)
                            st.markdown('**Slowest questions**')
                        cols=[k for k in ['paper_title','question',qm,'gold_count'] if k in hard]
                        st.dataframe(hard[cols],hide_index=True,use_container_width=True)
                    idx=st.number_input('Inspect question index',min_value=0,max_value=len(questions)-1,value=0)
                    q=questions.iloc[int(idx)]
                    st.write('**Question:**',q.get('question',''))
                    st.write('Gold source paragraph IDs:',q.get('gold_ids',[]))
                    st.write('Retrieved source paragraph IDs:',q.get('retrieved_ids',[]))
                    for j,ctx in enumerate(q.get('contexts',[]),1):
                        with st.expander(f'Retrieved passage {j}'):
                            st.write(ctx)

                st.markdown('#### Frequently missed questions across configurations')
                st.caption('Only questions evaluated by at least two selected runs appear. '
                           'Run coverage can differ because of missing evidence mappings.')
                counters=[]
                for name in runs[:100]:
                    qdf=run_questions(name)
                    if qdf.empty or not {'question_id','recall_at_k'}.issubset(qdf):continue
                    for _,r in qdf.iterrows():
                        counters.append({'question_id':str(r['question_id']),'question':r.get('question',''),
                                         'run':name,'recall':r['recall_at_k']})
                if counters:
                    cdf=pd.DataFrame(counters).groupby(['question_id','question'],as_index=False).agg(
                        n_runs=('run','nunique'),mean_recall=('recall','mean'),
                        missed_rate=('recall',lambda x:float((pd.to_numeric(x,errors='coerce')==0).mean())))
                    cdf=cdf[cdf['n_runs']>=2].sort_values(['missed_rate','n_runs'],ascending=[False,False])
                    st.dataframe(cdf.head(15),hide_index=True,use_container_width=True)

    for filename,title in [('generation_scores.csv','Local LLM groundedness / context relevance / answer relevance'),
                           ('ragas_scores.csv','Official RAGAS metrics')]:
        p=ART/filename
        if p.exists():
            st.subheader(title)
            try:
                llm_df=pd.read_csv(p)
                st.dataframe(llm_df,use_container_width=True,hide_index=True)
                candidate=[x for x in ['groundedness','context_relevance','answer_relevance',
                                        'faithfulness','answer_relevancy','context_precision','context_recall']
                           if x in llm_df and pd.to_numeric(llm_df[x],errors='coerce').notna().any()]
                if candidate:
                    long=llm_df[candidate].apply(pd.to_numeric,errors='coerce').melt(
                        var_name='metric',value_name='score').dropna()
                    chart=alt.Chart(long).mark_boxplot().encode(x='metric:N',y='score:Q',color='metric:N')
                    st.altair_chart(chart.properties(height=250),use_container_width=True)
                    st.caption('LLM-judge scores are shown separately from deterministic retrieval metrics.')
            except (OSError,ValueError) as exc:
                st.warning(f'Unable to read {filename}: {exc}')

with qa:
    st.info('Ask questions about one paper. Retrieval is local; Ollama is required for generated answers and query transformations.')
    try: papers=papers_cached(12)
    except Exception as exc:
        st.error(f'Cannot load QASPER: {exc}')
        st.stop()
    choices={f'{p.get("title",pid)} — {pid}':pid for pid,p in papers.items()}
    selected_title=st.selectbox('Paper',list(choices))
    runs=fetch_runs()
    if not runs.empty:
        chosen_run=st.selectbox('Use an evaluated experiment configuration', ['Manual']+runs['run'].tolist())
    else:chosen_run='Manual'
    if chosen_run!='Manual':
        record=runs[runs['run']==chosen_run].iloc[0].to_dict()
        cfg=Config(**{name:record[name] for name in Config.__dataclass_fields__ if name in record and pd.notna(record[name])})
        st.json({name:getattr(cfg,name) for name in ('chunking','retrieval','rerank','transformation','chunk_size','overlap','top_k','rrf_k')})
    else:
        a,b,c,d=st.columns(4)
        chunk=a.selectbox('Chunking',['paragraph','fixed','recursive','semantic','sentence_window','parent','auto_merging'])
        retrieval=b.selectbox('Retrieval',['dense','bm25','hybrid'])
        transform=c.selectbox('Query transform',['none','rewrite','multi_query','hyde'])
        rerank=d.checkbox('Cross-encoder reranking')
        a,b,c,d=st.columns(4)
        sz=a.number_input('Chunk size',64,2048,256,64)
        ov=b.number_input('Overlap',0,512,32,8)
        k=c.number_input('Top K',1,20,5)
        rrf=d.number_input('RRF k',1,200,60)
        cfg=Config(chunking=chunk,retrieval=retrieval,transformation=transform,rerank=rerank,
                   chunk_size=sz,overlap=ov,top_k=k,rrf_k=rrf)
    prompt=st.text_input('Your question','What are the main limitations of the proposed approach?')
    answer_on=st.checkbox('Generate response with local Ollama',value=False)
    if st.button('Retrieve / Answer',type='primary') and prompt.strip():
        if cfg.chunking in ('fixed','recursive') and cfg.overlap>=cfg.chunk_size:
            st.error('Overlap must be smaller than chunk size.')
        else:
            try:
                lab=search_cached(choices[selected_title],json.dumps(vars(cfg),sort_keys=True))
                started=time.perf_counter()
                hits=lab.search(prompt)
                st.caption(f'{len(hits)} retrieved passages in {time.perf_counter()-started:.2f}s')
                if answer_on:
                    st.subheader('Answer with citations')
                    st.write(generate_answer(prompt,hits,cfg))
                st.subheader('Retrieved evidence')
                for j,h in enumerate(hits,1):
                    with st.expander(f'[{j}] {h.node.metadata.get("section","Section")} · {h.node.metadata.get("source_pid", "unknown")}',expanded=j<=2):
                        st.write(passage(h,cfg))
            except Exception as exc:st.exception(exc)


Overwriting app.py


## 8. Launch Streamlit directly from Jupyter

The notebook writes `app.py`; Streamlit needs its own server process, but you do not need to maintain an external source file. Open http://localhost:8501 after running the launch cell.


In [14]:
import subprocess, sys
from IPython.display import display, HTML
if 'ragxiv_streamlit' not in globals() or ragxiv_streamlit.poll() is not None:
    ragxiv_streamlit=subprocess.Popen([
        sys.executable,'-m','streamlit','run','app.py',
        '--server.port','8501','--server.headless','true'
    ])
    print('Started Streamlit (PID:',ragxiv_streamlit.pid,')')
else:
    print('Streamlit already running (PID:',ragxiv_streamlit.pid,')')
display(HTML('<a href="http://localhost:8501" target="_blank">Open RAGxiv dashboard</a>'))


Started Streamlit (PID: 82343 )


2026-11-24 12:07:05.573 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.1.19:8501
  External URL: http://171.61.166.207:8501

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            


2026-11-24 12:24:50.027 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'` or specify an integer width.
2026-11-24 12:24:50.027 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-11-24 12:24:50.058 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'` or specify an integer width.
2026-11-24 12:24:50.059 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width

## Notes on interpreting experiment results

- QASPER gold evidence is originally annotated at the paragraph level. Hits are mapped to `source_pid` for all chunkers. Thus chunk sizes affect retrieval, but retrieved duplicate chunks do not inflate gold evidence recall.
- Retrieval-only runs compute Precision@K, Recall@K, Hit@K, MRR, MAP, nDCG, latency, and mapping coverage. `precision_at_k` divides by K even if fewer than K unique source paragraphs are retrieved.
- Parent-document and auto-merging are **simplified** implementations, not exact LlamaIndex AutoMergingRetriever equivalents.
- Single-paper retrieval is the baseline; cross-paper discovery requires a different benchmark.
- Query transformations use local Ollama when enabled, and may be slow. No paid web search is used.